In [1]:
spark.stop()

NameError: name 'spark' is not defined

In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/Kearney/prathyusha')
from utils import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = instantiate_spark_sedona("10g")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/18 11:04:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/18 11:04:55 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
26/02/18 11:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/18 11:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/18 11:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/18 11:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/02/18 11:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


Spark Session and SedonaContext have been successfully initiated.


In [2]:
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

In [3]:
loc = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("abfss://propheus-data-staging@propheusdatabay.dfs.core.windows.net/kearney/propheus-rtb-data/sample-data/*")
)

loc = loc.withColumn("timestamp_ist", to_timestamp(col("timestamp") / 1000))

loc.show(5)


26/02/18 11:05:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+--------------------+------------+----+------------------+-----------------+------------+-------+-------+-----------+----------+------------+------+-----------+-------+-----------------+------+------------------+-------+----------+--------+----------------+--------------------+--------------------+-----+--------------------+-----------------+----------+--------------+--------------------+
|    timestamp|                maid|        ipv4|ipv6|          latitude|        longitude|country_code| region|   city|    carrier|connection|network_type|mccmnc|        isp|   make|            model|device|             brand|     os|os_version|language|          bundle|            app_name|                  ua| gdpr|carrier_country_code|     carrier_name|javascript|consent_string|       timestamp_ist|
+-------------+--------------------+------------+----+------------------+-----------------+------------+-------+-------+-----------+----------+------------+------+-----------+-------+---

In [4]:
loc.count()

205106806

In [5]:
loc.printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- maid: string (nullable = true)
 |-- ipv4: string (nullable = true)
 |-- ipv6: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- country_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- connection: string (nullable = true)
 |-- network_type: string (nullable = true)
 |-- mccmnc: string (nullable = true)
 |-- isp: string (nullable = true)
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- device: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- os: string (nullable = true)
 |-- os_version: string (nullable = true)
 |-- language: string (nullable = true)
 |-- bundle: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- ua: string (nullable = true)
 |-- gdpr: boolean (nullable = true)
 |-- carrier_country_code: string (nul

In [16]:
loc_with_date.select(col('event_date')).distinct().show()

+----------+
|event_date|
+----------+
|2025-12-05|
|2025-12-04|
|2025-12-31|
|2026-01-01|
|2025-12-03|
+----------+



In [6]:
buil = spark.read.parquet('s3a://propheus-data-staging/Thailand/Gee_Buildings/overture_tiles_merged_buildings.parquet')
buil.count()

26/02/10 04:40:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


34774205

In [7]:
buil.printSchema()

root
 |-- geom_source_1: string (nullable = true)
 |-- geom_source_2: geometry (nullable = true)
 |-- iou: double (nullable = true)
 |-- id_a: long (nullable = true)
 |-- id_b: long (nullable = true)
 |-- height_tiles: string (nullable = true)
 |-- height_overture: double (nullable = true)
 |-- num_floors_overture: integer (nullable = true)
 |-- subtype_overture: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- province_en: string (nullable = true)
 |-- district_en: string (nullable = true)
 |-- tambon_en: string (nullable = true)
 |-- tambon_code: string (nullable = true)
 |-- gh6: string (nullable = true)



In [9]:
buil_geo = buil.select(
    col("id_a"),
    expr("ST_GeomFromWKT(geom_source_1)").alias("buil_geom")
).repartition(200)

# Enable spatial index
spark.conf.set("sedona.global.index", "true")
spark.conf.set("sedona.global.indextype", "rtree")


loc_with_date = loc.withColumn(
    "event_date",
    to_date(col("timestamp_ist"))
)



dates = [r.event_date for r in loc_with_date.select("event_date").distinct().collect()]

print(f"Processing {len(dates)} days")


for d in dates:

    print(f"Processing {d}")

    loc_day = (
        loc_with_date.drop("consent_string")
        .filter(col("event_date") == d)
        .withColumn("point_geom", expr("ST_Point(longitude, latitude)"))
        .repartition(300)
    )

    joined_day = loc_day.join(
        buil_geo,
        expr("ST_Within(point_geom, buil_geom)"),
        "inner"
    ).drop("point_geom")



    joined_day.write.mode("overwrite").parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Buildings_w_rtb_dec_loc_data')

print("✅ All days processed")


Processing 5 days
Processing 2025-12-05


26/02/10 04:42:12 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.


[194.530s][warning][gc,alloc] Executor task launch worker for task 53.0 in stage 24.0 (TID 1205): Retried waiting for GCLocker too often allocating 4018 words
[194.549s][warning][gc,alloc] Executor task launch worker for task 87.0 in stage 24.0 (TID 1239): Retried waiting for GCLocker too often allocating 37080 words
[194.549s][warning][gc,alloc] Executor task launch worker for task 74.0 in stage 24.0 (TID 1226): Retried waiting for GCLocker too often allocating 16386 words
[194.561s][warning][gc,alloc] Executor task launch worker for task 78.0 in stage 24.0 (TID 1230): Retried waiting for GCLocker too often allocating 27947 words
[194.561s][warning][gc,alloc] Executor task launch worker for task 61.0 in stage 24.0 (TID 1213): Retried waiting for GCLocker too often allocating 6571 words
[194.561s][warning][gc,alloc] Executor task launch worker for task 60.0 in stage 24.0 (TID 1212): Retried waiting for GCLocker too often allocating 3825 words
[194.561s][warning][gc,alloc] Executor task

26/02/10 04:42:21 ERROR Executor: Exception in task 74.0 in stage 24.0 (TID 1226)
java.lang.OutOfMemoryError: Java heap space
	at scala.reflect.ManifestFactory$ObjectManifest.newArray(Manifest.scala:258)
	at scala.reflect.ManifestFactory$ObjectManifest.newArray(Manifest.scala:257)
	at org.apache.spark.util.collection.KVArraySortDataFormat.allocate(SortDataFormat.scala:110)
	at org.apache.spark.util.collection.KVArraySortDataFormat.allocate(SortDataFormat.scala:86)
	at org.apache.spark.util.collection.TimSort$SortState.ensureCapacity(TimSort.java:943)
	at org.apache.spark.util.collection.TimSort$SortState.mergeHi(TimSort.java:811)
	at org.apache.spark.util.collection.TimSort$SortState.mergeAt(TimSort.java:519)
	at org.apache.spark.util.collection.TimSort$SortState.mergeCollapse(TimSort.java:445)
	at org.apache.spark.util.collection.TimSort$SortState.access$200(TimSort.java:308)
	at org.apache.spark.util.collection.TimSort.sort(TimSort.java:136)
	at org.apache.spark.util.collection.Sorte

Py4JJavaError: An error occurred while calling o321.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 74 in stage 24.0 failed 1 times, most recent failure: Lost task 74.0 in stage 24.0 (TID 1226) (dst-data-compute.internal.cloudapp.net executor driver): java.lang.OutOfMemoryError: Java heap space
	at scala.reflect.ManifestFactory$ObjectManifest.newArray(Manifest.scala:258)
	at scala.reflect.ManifestFactory$ObjectManifest.newArray(Manifest.scala:257)
	at org.apache.spark.util.collection.KVArraySortDataFormat.allocate(SortDataFormat.scala:110)
	at org.apache.spark.util.collection.KVArraySortDataFormat.allocate(SortDataFormat.scala:86)
	at org.apache.spark.util.collection.TimSort$SortState.ensureCapacity(TimSort.java:943)
	at org.apache.spark.util.collection.TimSort$SortState.mergeHi(TimSort.java:811)
	at org.apache.spark.util.collection.TimSort$SortState.mergeAt(TimSort.java:519)
	at org.apache.spark.util.collection.TimSort$SortState.mergeCollapse(TimSort

26/02/10 04:42:23 ERROR IndexShuffleBlockResolver: Failed to write checksum file
java.io.FileNotFoundException: /tmp/blockmgr-3710f38e-7372-4a46-b4d1-b101116756cd/2b/shuffle_6_1213_0.checksum.ADLER32.b78de5cb-39e4-4a0f-b9d1-3ea4cadf092a (No such file or directory)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:184)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.writeMetadataFile(IndexShuffleBlockResolver.scala:455)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.$anonfun$writeMetadataFileAndCommit$3(IndexShuffleBlockResolver.scala:402)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.$anonfun$writeMetadataFileAndCommit$3$adapted(IndexShuffleBlockResolver.scala:400)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark